In [38]:
from datasets import load_dataset

dataset = load_dataset("wikitext", "wikitext-2-v1")
print(dataset.keys())   
print(dataset['train'][:5])

dict_keys(['test', 'train', 'validation'])
{'text': ['', ' = Valkyria Chronicles III = \n', '', ' Senjō no Valkyria 3 : <unk> Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . <unk> the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " <unk> Raven " . \n', " The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustme

In [40]:
bias_data = load_dataset("oskarvanderwal/winogender", "all")["test"]
print(bias_data[0])

{'sentid': 'technician.customer.1.male.txt', 'sentence': 'The technician told the customer that he could pay with cash.', 'pronoun': 'he', 'occupation': 'technician', 'participant': 'customer', 'gender': 'male', 'target': 'customer', 'label': 1}


In [39]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2", use_fast=True)
tokenizer.pad_token = tokenizer.eos_token


In [41]:
def tokenize_batch(batch):
    return tokenizer(batch["text"], return_special_tokens_mask=False)

tokenized_train = dataset["train"].map(tokenize_batch, batched=True, remove_columns=["text"])
tokenized_val   = dataset["validation"].map(tokenize_batch, batched=True, remove_columns=["text"])

if "attention_mask" in tokenized_train.column_names:
    tokenized_train = tokenized_train.remove_columns(["attention_mask"])
if "attention_mask" in tokenized_val.column_names:
    tokenized_val = tokenized_val.remove_columns(["attention_mask"])


Map: 100%|██████████| 3760/3760 [00:00<00:00, 25549.21 examples/s]


In [42]:
block_size = 256

def group_texts(batch):
    concatenated = []
    for tokens in batch["input_ids"]:
        concatenated.extend(tokens)
    total_length = (len(concatenated) // block_size) * block_size
    concatenated = concatenated[:total_length]
    chunks = [concatenated[i : i + block_size] for i in range(0, total_length, block_size)]
    return {
        "input_ids": chunks,
        "labels": chunks
    }

lm_train = tokenized_train.map(group_texts, batched=True, batch_size=1000)
lm_val   = tokenized_val.map(group_texts, batched=True, batch_size=1000)
print(lm_train[0]["input_ids"][:10])

Map:   0%|          | 0/36718 [00:00<?, ? examples/s]

Map: 100%|██████████| 3760/3760 [00:00<00:00, 50818.42 examples/s]

[796, 569, 18354, 7496, 17740, 6711, 796, 220, 198, 2311]


In [43]:
def tokenize_sentence(batch):
    return tokenizer(batch["sentence"], truncation=True)

tokenized_bias = bias_data.map(tokenize_sentence, batched=True, remove_columns=bias_data.column_names)
tokenized_bias = tokenized_bias.map(lambda x: {"labels": x["input_ids"]})

bias_block_size = max(len(x) for x in tokenized_bias["input_ids"])

def pad_batch(batch):
    padded = tokenizer.pad(
        {
            "input_ids": batch["input_ids"],
            "labels" : batch["labels"],
        },
        padding="max_length",
        max_length=bias_block_size,
    )
    return padded



tokenized_bias = tokenized_bias.map(pad_batch, batched=True)
tokenized_bias.set_format(type="torch", columns=["input_ids", "labels"])


Map: 100%|██████████| 720/720 [00:00<00:00, 46457.84 examples/s]


In [44]:
from transformers import GPT2Config, GPT2LMHeadModel

config = GPT2Config(
    vocab_size=tokenizer.vocab_size,
    n_positions=block_size,
    n_ctx=block_size,
    n_embd=768,
    n_layer=12,
    n_head=12,
)
model = GPT2LMHeadModel(config)


In [45]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="../models/gpt2-small-wikitext",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1, 
    learning_rate=5e-4,
    warmup_steps=500,               
    lr_scheduler_type="cosine",     
    fp16=True,                       
    logging_steps=100,
    logging_dir="./logs",
    save_steps=500,
    save_total_limit=2,
    evaluation_strategy="epoch",
)

trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=lm_train,
    eval_dataset=lm_val,
)
trainer.train()


/Users/charanganesh/miniforge3/envs/AI/lib/python3.10/site-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/var/folders/k1/pg7cnkbs3fzdgsq32r3ht2_80000gn/T/ipykernel_81286/1753286836.py:20: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


ValueError: fp16 mixed precision requires a GPU (not 'mps').

In [ ]:
trainer.save_model("../models/gpt2_small_wikitext_model")

In [ ]:
from transformers import GPT2LMHeadModel

base_model = GPT2LMHeadModel.from_pretrained("../models/gpt2_small_wikitext_model").cuda()


bias_training_args = TrainingArguments(
    output_dir="./gpt2-small-debiased",
    overwrite_output_dir=True,
    num_train_epochs=5,
    per_device_train_batch_size=16,
    learning_rate=5e-5,
    fp16=True,
    logging_steps=10,
    save_steps=50,
    save_total_limit=1,
)

bias_trainer = Trainer(
    model=base_model,
    tokenizer=tokenizer,
    args=bias_training_args,
    train_dataset=tokenized_bias,
)
bias_trainer.train()

In [ ]:
bias_trainer.save_model("../models/gpt2_small_debiased_model")

In [ ]:
from transformers import pipeline

debiased_generator = pipeline(
    "text-generation",
    model="./gpt2_small_debiased_model",
    tokenizer=tokenizer,
    device=0
)

def generate_text(prompt, max_length=50):
    outputs = debiased_generator(prompt, max_new_tokens=max_length, do_sample=True, top_k=50, num_return_sequences=1)
    print(outputs[0]["generated_text"])


prompt1 = "The nurse told the doctor that"
prompt2 = "The software engineer laughed because"
print("Prompt:", prompt1)
generate_text(prompt1)
print("\nPrompt:", prompt2)
generate_text(prompt2)

In [ ]:
user_prompt = input("Enter a prompt: ")
generate_text(user_prompt)

In [ ]:
import torch